# Type Conversion and Coercion

## Overview
Type conversion is the process of changing a value from one data type to another. In SQL it happens two ways:

**Explicit:** `CAST(expression AS type)` or SQLite's `TYPEOF()`

**Implicit (coercion):** The database converts automatically — sometimes usefully, sometimes silently wrong.

**SQLite type affinity** (unique to SQLite):
SQLite uses a flexible type system. Every value has one of five storage classes: `NULL`, `INTEGER`, `REAL`, `TEXT`, `BLOB`. Columns have "affinity" (a preference), but any value can be stored in any column. This means:
- `'123'` stored in an INTEGER affinity column is coerced to `123`
- `123` stored in a TEXT affinity column becomes `'123'`
- Comparisons between `'10'` and `'9'` are lexicographic (TEXT) unless cast

**PostgreSQL is stricter:** Types are enforced at the column level. Implicit coercions are narrower — `'123'::INTEGER` (cast syntax) or `CAST('123' AS INTEGER)` is required for most conversions.

**CAST targets in SQLite:** `INTEGER`, `REAL`, `TEXT`, `NUMERIC`, `BLOB`

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE intake_raw (record_id INTEGER PRIMARY KEY, patient_ref TEXT, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, phone TEXT, email TEXT, cost_str TEXT, intake_date TEXT, notes TEXT);
CREATE TABLE lab_raw (lab_id INTEGER PRIMARY KEY, patient_ref TEXT, test_code TEXT, result_txt TEXT, collected TEXT, status TEXT);
CREATE TABLE provider_raw (prov_id INTEGER PRIMARY KEY, name_raw TEXT, dept_code TEXT, start_dt TEXT, salary TEXT);
INSERT INTO intake_raw VALUES
  (1,'P-001','aroha','Ngata','1985-03-12','F','NB','506-555-0101','aroha@mail.com','$3,200.00','2023-01-15','Referred by GP'),
  (2,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (3,'P-003','FATIMA','AL-RASHID','1990-07-22','female','Ontario','416-555-0303',NULL,'120.00','2023-03-05','Annual checkup'),
  (4,'P-004','James','MacLeod','Jan 30 1955','M','NB',NULL,'james@mail.com','$5,500','18-03-2023','Surgery follow-up'),
  (5,'P-005','sofia','Petrov','2001-09-15','F','BC','604-555-0505','sofia@mail.com','$95.00','2023-04-02',NULL),
  (6,'P-006','Noah','Williams','08/06/1968','m','AB ','780-555-0606','noah@mail.com','780','11/05/2023',NULL),
  (7,'P-007','Mei','Zhang','1995-04-17','F','ON','416-555-0707',NULL,'$2,100.00','22-06-2023','Follow-up required'),
  (8,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (9,'P-008','ethan','tremblay','01/12/1980',NULL,'QC','514-555-0808','ethan@mail.com','80.00','14-07-2023',NULL),
  (10,'P-009','Priya','Nair','1977-08-25','F','bc',NULL,'priya@mail.com','$1,750.00','01/10/2023',NULL),
  (11,'P-010','Marcus','Okafor','1993-05-19','M','ON','647-555-1010',NULL,'2200','03-11-2023',NULL),
  (12,'P-011','Diana','Flores','14/02/2000','female','NB','506-555-1111','diana@mail.com',NULL,'2023-12-01',NULL),
  (13,NULL,'Unknown',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,'Incomplete record'),
  (14,'','Test','Record','2023-01-01','X','XX','000-000-0000','test@test.com','-1','2023-01-01','Test entry');
INSERT INTO lab_raw VALUES
  (1,'P-001','HBA1C','7.2 %','2023-03-10','Active'),
  (2,'P-002','HBA1C','9.1%','2023-03-15','active'),
  (3,'P-003','CREAT','88.5umol/L','2023-04-01','ACTIVE'),
  (4,'P-004','CREAT','145 umol/L','2023-04-12','Active'),
  (5,'P-005','HBA1C','5.8 %','2023-05-05',''),
  (6,'P-006','SODIUM','138mmol/L','2023-05-20','Active'),
  (7,'P-007','SODIUM','151 mmol/L','2023-06-01',NULL),
  (8,'P-001','CREAT','72.3 umol/L','2023-06-18','Active'),
  (9,'P-008','HBA1C','6.4%','2023-07-02','Active'),
  (10,'P-009','CREAT','210umol/L','2023-07-15','active');
INSERT INTO provider_raw VALUES
  (1,'MacNeil, Sarah MD','CARD','2018-01-15','$95,000'),
  (2,'Dr. James Wong','ONCO','2019-03-01','88000'),
  (3,'Fatima Osei M.D.','FAM','2017-06-01','$82,500.00'),
  (4,'Larson, Ethan','ORTH','2020-09-15','91000.00'),
  (5,'Sharma, Priya MD','EMRG','2016-11-01','$78,000'),
  (6,'Lucas Petit','RAD','2021-02-28',NULL);
""")
conn.commit()
print("Schema ready.")

Schema ready.


---
## Explicit CAST — numeric strings to numbers

In [2]:
# cost_str contains "$3,200.00", "1850", "$5,500", "-1" etc.
# Must strip $, commas, then cast
print("=== Cleaning cost_str and casting to REAL ===")
q = """
SELECT  record_id,
        cost_str,
        CAST(
            REPLACE(REPLACE(COALESCE(cost_str,'0'),'$',''),',','')
        AS REAL)                             AS cost_numeric,
        TYPEOF(cost_str)                     AS original_type,
        TYPEOF(CAST(
            REPLACE(REPLACE(COALESCE(cost_str,'0'),'$',''),',','')
        AS REAL))                            AS cast_type
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

=== Cleaning cost_str and casting to REAL ===
 record_id  cost_str  cost_numeric original_type cast_type
         1 $3,200.00        3200.0          text      real
         2      1850        1850.0          text      real
         3    120.00         120.0          text      real
         4    $5,500        5500.0          text      real
         5    $95.00          95.0          text      real
         6       780         780.0          text      real
         7 $2,100.00        2100.0          text      real
         8      1850        1850.0          text      real
         9     80.00          80.0          text      real
        10 $1,750.00        1750.0          text      real
        11      2200        2200.0          text      real
        12      None           0.0          null      real
        13      None           0.0          null      real
        14        -1          -1.0          text      real


---
## The lexicographic sort trap — text numbers sort wrong

In [3]:
# Demonstrate that sorting text numbers gives wrong order
print("=== Text vs numeric sort order (the classic trap) ===")
conn.executescript("""
CREATE TEMP TABLE cost_test AS
SELECT record_id, REPLACE(REPLACE(COALESCE(cost_str,'0'),'$',''),',','') AS cost_text
FROM intake_raw WHERE cost_str IS NOT NULL AND CAST(REPLACE(REPLACE(cost_str,'$',''),',','') AS REAL) > 0;
""")

q_text = "SELECT record_id, cost_text FROM cost_test ORDER BY cost_text ASC"
q_num  = "SELECT record_id, cost_text FROM cost_test ORDER BY CAST(cost_text AS REAL) ASC"
print("Sorted as TEXT (wrong order for numbers):")
print(pd.read_sql(q_text, conn).to_string(index=False))
print()
print("Sorted as REAL (correct):")
print(pd.read_sql(q_num, conn).to_string(index=False))

=== Text vs numeric sort order (the classic trap) ===
Sorted as TEXT (wrong order for numbers):
 record_id cost_text
         3    120.00
        10   1750.00
         2      1850
         8      1850
         7   2100.00
        11      2200
         1   3200.00
         4      5500
         6       780
         9     80.00
         5     95.00

Sorted as REAL (correct):
 record_id cost_text
         9     80.00
         5     95.00
         3    120.00
         6       780
        10   1750.00
         2      1850
         8      1850
         7   2100.00
        11      2200
         1   3200.00
         4      5500


---
## Salary cleaning — provider_raw

In [4]:
# salary column: "$95,000", "88000", "$82,500.00", NULL
print("=== provider_raw salary: strip symbols and cast ===")
q = """
SELECT  prov_id,
        name_raw,
        salary,
        CAST(
            REPLACE(REPLACE(REPLACE(COALESCE(salary,'0'),'$',''),',',''),' ','')
        AS REAL)                          AS salary_num,
        CASE
            WHEN salary IS NULL THEN 'Missing'
            WHEN CAST(REPLACE(REPLACE(salary,'$',''),',','') AS REAL) < 80000 THEN 'Band 1'
            WHEN CAST(REPLACE(REPLACE(salary,'$',''),',','') AS REAL) < 90000 THEN 'Band 2'
            ELSE 'Band 3'
        END AS salary_band
FROM    provider_raw
ORDER BY salary_num DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# PostgreSQL cast syntax note
print()
print("PostgreSQL alternative cast syntaxes:")
print("  CAST(col AS INTEGER)      -- ANSI standard, works everywhere")
print("  col::INTEGER              -- PostgreSQL shorthand")
print("  TO_NUMBER(col, '99,999') -- PostgreSQL: pattern-based numeric parse")

conn.close()

=== provider_raw salary: strip symbols and cast ===
 prov_id          name_raw     salary  salary_num salary_band
       1 MacNeil, Sarah MD    $95,000     95000.0      Band 3
       4     Larson, Ethan   91000.00     91000.0      Band 3
       2    Dr. James Wong      88000     88000.0      Band 2
       3  Fatima Osei M.D. $82,500.00     82500.0      Band 2
       5  Sharma, Priya MD    $78,000     78000.0      Band 1
       6       Lucas Petit       None         0.0     Missing

PostgreSQL alternative cast syntaxes:
  CAST(col AS INTEGER)      -- ANSI standard, works everywhere
  col::INTEGER              -- PostgreSQL shorthand
  TO_NUMBER(col, '99,999') -- PostgreSQL: pattern-based numeric parse


---
## Common Pitfalls

**1. Sorting numeric values stored as text gives wrong results**
`ORDER BY cost_text` where values are `'95', '1200', '88'` sorts as `'1200', '88', '95'` (lexicographic). Always `CAST(col AS REAL)` or `CAST(col AS INTEGER)` before sorting or comparing numeric strings.

**2. CAST on a non-numeric string silently returns 0 in SQLite**
`CAST('abc' AS INTEGER)` returns `0` in SQLite. In PostgreSQL it raises an error. Use `TYPEOF()` and validation checks to catch non-numeric values before casting in production pipelines.

**3. Stripping only $ but forgetting commas**
`CAST(REPLACE(cost_str,'$','') AS REAL)` on `'$1,200'` returns `1.2` (stops parsing at the comma). Always strip both `$` and `,` before casting monetary strings.

**4. CAST(NULL AS INTEGER) returns NULL — not 0**
`CAST(NULL AS REAL)` is NULL. Wrap with `COALESCE(CAST(col AS REAL), 0)` when you need a numeric default.

**5. SQLite implicit coercion can mask type mismatches**
SQLite will silently compare `'10' = 10` as TRUE in some contexts. PostgreSQL raises a type mismatch error. Write explicit CASTs for portability — don't rely on SQLite's permissive type system in code that may run on PostgreSQL.

**6. Integer division loses the decimal in some contexts**
In SQLite, `CAST(5 AS INTEGER) / CAST(2 AS INTEGER)` returns `2` (integer division). Use `CAST(5 AS REAL) / 2` or `5 * 1.0 / 2` to force floating-point division.


---
*sql_methods_library - Samantha McGarrigle*